# 🚀 Reportes DAO — Demo y pruebas
**Proyecto:** CivicTech  
**Objetivo:** demostrar creación, consulta y gestión de reportes con evidencias usando `ReportesDAO`.  
**Entorno:** Jupyter Notebook · Python · MongoDB (ajustar `MONGO_URI` en la celda de conexión)  

**Instrucciones rápidas**
- Activá el entorno virtual: `source .venv/bin/activate`  
- Asegurate de que MongoDB esté accesible desde `MONGO_URI`.  
- Si la importación falla, usá `from dao.reporte_dao import ReportesDAO` o añadí `from .reporte_dao import ReportesDAO` en `dao/__init__.py`.


In [1]:
# Celda: inicializar MongoDAO y dejar `mongo` disponible
from dao.mongo_dao import MongoDAO
mongo = MongoDAO()
db = mongo.connect()
print("Conectado a DB:", mongo.db.name)

Conectado a DB: civictech


In [2]:
# Celda: inicializar ReportesDAO (ejecutá solo esta)
from dao.reporte_dao import ReportesDAO
import pprint
pp = pprint.PrettyPrinter(indent=2)

reportes_dao = ReportesDAO(mongo)
print("DAO inicializado. Colección reporte:", reportes_dao.reporte_col.name)



DAO inicializado. Colección reporte: reporte


In [3]:
# Celda 3 corregida: asegurar que `mongo` esté definido y conectado (ejecutá solo esta)
from dao.mongo_dao import MongoDAO

try:
    mongo  # si ya existe, no hacemos nada
except NameError:
    mongo = MongoDAO()
    db = mongo.connect()
    print("Conectado a DB:", mongo.db.name)
else:
    # comparar explícitamente con None para evitar NotImplementedError
    if getattr(mongo, "db", None) is None:
        db = mongo.connect()
        print("Conectado a DB:", mongo.db.name)
    else:
        # mongo.db existe; mostrar su nombre sin evaluar su truthiness
        try:
            print("Variable 'mongo' ya definida. DB:", mongo.db.name)
        except Exception as e:
            print("Variable 'mongo' definida pero no se pudo leer mongo.db.name:", type(e).__name__, str(e))


Variable 'mongo' ya definida. DB: civictech


### 🧾 Insertar reporte de prueba — resumen en tabla

**Qué hace:** Inserta un reporte con evidencia usando `ReportesDAO` y muestra un resumen compacto en formato tabla.

In [4]:
# Celda: insertar reporte con ReportesDAO y mostrar salida en tabla clásica
from datetime import datetime, timezone
from dao.reporte_dao import ReportesDAO

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=60):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# Instanciar DAO
reportes_dao = ReportesDAO(mongo)

# Tomar primer usuario disponible
usuario = mongo.db.usuario.find_one({}, {"_id":1, "municipio_id":1})
if not usuario:
    raise RuntimeError("No se encontró ningún usuario en la colección 'usuario'.")

usuario_id = usuario["_id"]
municipio_id = usuario.get("municipio_id")
if not municipio_id:
    raise RuntimeError("El usuario seleccionado no tiene 'municipio_id' asignado.")

now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "DEMO123",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-67.49, -29.16]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_demo_001",
    "descripcion": "Vehículo bloqueando rampa de acceso peatonal. Foto adjunta como evidencia."
}

evidencia_doc = {
    "url_foto": "https://gcdn.emol.cl/argentina/files/2016/05/auto-tapando-rampa.jpg",
    "hash_seguridad_sha": "sha256:demo_001",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Llamada al DAO (wrapper)
reporte_id, evidencia_id, err = reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)

if err:
    print("Error al insertar:", err)
else:
    # Recuperar documentos para mostrar
    rpt = reportes_dao.get_reporte_by_id(reporte_id)
    ev = reportes_dao.evidencia_col.find_one({"reporte_id": reporte_id})
    mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1})
    municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"

    # Preparar tabla clásica (anchos ajustables)
    cols = [
        ("Campo", 18),
        ("Valor", 70)
    ]
    sep = "+" + "+".join(["-"*w for _, w in cols]) + "+"

    def row(key, val):
        k = f" {key}".ljust(cols[0][1])
        v = f" {val}".ljust(cols[1][1])
        return f"|{k}|{v}|"

    print(sep)
    print(row("reporte_id", safe_str(rpt.get("_id"))))
    print(sep)
    print(row("patente_vehiculo", safe_str(rpt.get("patente_vehiculo"))))
    print(row("estado", safe_str(rpt.get("estado"))))
    print(row("fechaHora_server", safe_str(rpt.get("fechaHora_server"))))
    print(row("usuario_id", safe_str(rpt.get("usuario_id"))))
    print(row("municipio_id", safe_str(rpt.get("municipio_id"))))
    print(row("municipio_nombre", safe_str(municipio_nombre)))
    print(row("ubicacion", safe_str(rpt.get("ubicacion"))))
    print(row("descripcion", safe_str(rpt.get("descripcion"), maxlen=200)))
    print(sep)
    print(row("evidencia_id", safe_str(ev.get("_id") if ev else None)))
    print(row("url_foto", safe_str(ev.get("url_foto") if ev else None)))
    print(row("hash_seguridad_sha", safe_str(ev.get("hash_seguridad_sha") if ev else None)))
    print(row("uploaded_by", safe_str(ev.get("uploaded_by") if ev else None)))
    print(row("uploaded_at", safe_str(ev.get("uploaded_at") if ev else None)))
    print(sep)


+------------------+----------------------------------------------------------------------+
| reporte_id       | 6a28f45ca8d851a602aefa36                                             |
+------------------+----------------------------------------------------------------------+
| patente_vehiculo | DEMO123                                                              |
| estado           | Pendiente                                                            |
| fechaHora_server | 2026-06-10 05:21:32.908000                                           |
| usuario_id       | 6a28b11e3ca8ed0b349df8a6                                             |
| municipio_id     | 6a28b11e3ca8ed0b349df8a4                                             |
| municipio_nombre | Chilecito                                                            |
| ubicacion        | {'type': 'Point', 'coordinates': [-67.49, -29.16]}                   |
| descripcion      | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta 

In [5]:
# Celda: Insertar reporte en Famatina (usuario: Jorge Vera) — "auto estacionado en la vereda"

from datetime import datetime, timezone
from bson import ObjectId
from dao.reporte_dao import ReportesDAO

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=80):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

reportes_dao = ReportesDAO(mongo)

# IDs conocidos
USUARIO_ID = ObjectId("6a28b11e3ca8ed0b349df8ae")   # Jorge Vera (Famatina)
MUNICIPIO_ID = ObjectId("6a28b11e3ca8ed0b349df8a3") # Famatina

usuario = mongo.db.usuario.find_one({"_id": USUARIO_ID}, {"_id":1, "municipio_id":1, "nombre_apellido":1})
if not usuario:
    raise RuntimeError("No se encontró el usuario especificado (Jorge Vera).")

usuario_id = usuario["_id"]
usuario_nombre = usuario.get("nombre_apellido", "NOMBRE_NO_REGISTRADO")
now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": MUNICIPIO_ID,
    "patente_vehiculo": "FAM-VERA-01",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-67.85, -29.21]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_fam_vera_001",
    "descripcion": "Auto estacionado en la vereda"
}

evidencia_doc = {
    "url_foto": "https://gcdn.emol.cl/argentina/files/2016/05/auto-tapando-rampa.jpg",
    "hash_seguridad_sha": "sha256:fam_vera_001",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

reporte_id, evidencia_id, err = reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)

if err:
    print("Error al insertar:", err)
else:
    rpt = reportes_dao.get_reporte_by_id(reporte_id)
    ev = reportes_dao.evidencia_col.find_one({"reporte_id": reporte_id})
    mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1, "codigo_municipio":1})
    municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"
    municipio_codigo = mun.get("codigo_municipio") if mun else ""

    cols = [("Campo", 20), ("Valor", 70)]
    sep = "+" + "+".join(["-"*w for _, w in cols]) + "+"

    def row(key, val):
        k = f" {key}".ljust(cols[0][1])
        v = f" {val}".ljust(cols[1][1])
        return f"|{k}|{v}|"

    print(sep)
    print(row("reporte_id", safe_str(rpt.get("_id"))))
    print(sep)
    print(row("usuario_id", safe_str(rpt.get("usuario_id"))))
    print(row("usuario_nombre", safe_str(usuario_nombre)))
    print(row("municipio_id", safe_str(rpt.get("municipio_id"))))
    print(row("municipio_nombre", safe_str(municipio_nombre)))
    print(row("municipio_codigo", safe_str(municipio_codigo)))
    print(row("patente_vehiculo", safe_str(rpt.get("patente_vehiculo"))))
    print(row("estado", safe_str(rpt.get("estado"))))
    print(row("fechaHora_server", safe_str(rpt.get("fechaHora_server"))))
    print(row("ubicacion", safe_str(rpt.get("ubicacion"))))
    print(row("descripcion", safe_str(rpt.get("descripcion"), maxlen=200)))
    print(sep)
    print(row("evidencia_id", safe_str(ev.get("_id") if ev else None)))
    print(row("url_foto", safe_str(ev.get("url_foto") if ev else None)))
    print(row("hash_seguridad_sha", safe_str(ev.get("hash_seguridad_sha") if ev else None)))
    print(row("uploaded_by", safe_str(ev.get("uploaded_by") if ev else None)))
    print(row("uploaded_at", safe_str(ev.get("uploaded_at") if ev else None)))
    print(sep)


+--------------------+----------------------------------------------------------------------+
| reporte_id         | 6a28f45ea8d851a602aefa38                                             |
+--------------------+----------------------------------------------------------------------+
| usuario_id         | 6a28b11e3ca8ed0b349df8ae                                             |
| usuario_nombre     | Jorge Vera                                                           |
| municipio_id       | 6a28b11e3ca8ed0b349df8a3                                             |
| municipio_nombre   | Famatina                                                             |
| municipio_codigo   | FAM001                                                               |
| patente_vehiculo   | FAM-VERA-01                                                          |
| estado             | Pendiente                                                            |
| fechaHora_server   | 2026-06-10 05:21:34.221000           

In [6]:
# Celda 2: Insertar reporte en Famatina (usuario: Elena Paz) — "auto abandonado"
from datetime import datetime, timezone
from bson import ObjectId
from dao.reporte_dao import ReportesDAO

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=80):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

reportes_dao = ReportesDAO(mongo)

# IDs conocidos
USUARIO_ID = ObjectId("6a28b11e3ca8ed0b349df8af")   # Elena Paz (Famatina)
MUNICIPIO_ID = ObjectId("6a28b11e3ca8ed0b349df8a3") # Famatina

usuario = mongo.db.usuario.find_one({"_id": USUARIO_ID}, {"_id":1, "municipio_id":1, "nombre_apellido":1})
if not usuario:
    raise RuntimeError("No se encontró el usuario especificado (Elena Paz).")

usuario_id = usuario["_id"]
usuario_nombre = usuario.get("nombre_apellido", "NOMBRE_NO_REGISTRADO")
now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": MUNICIPIO_ID,
    "patente_vehiculo": "FAM-ELENA-02",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-67.84, -29.205]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_fam_elena_002",
    "descripcion": "Auto abandonado"
}

evidencia_doc = {
    "url_foto": "https://fotos.perfil.com/2020/05/07/autos-mal-estacionados-20200507-952139.jpg",
    "hash_seguridad_sha": "sha256:fam_elena_002",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

reporte_id, evidencia_id, err = reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)

if err:
    print("Error al insertar:", err)
else:
    rpt = reportes_dao.get_reporte_by_id(reporte_id)
    ev = reportes_dao.evidencia_col.find_one({"reporte_id": reporte_id})
    mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1, "codigo_municipio":1})
    municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"
    municipio_codigo = mun.get("codigo_municipio") if mun else ""

    cols = [("Campo", 20), ("Valor", 70)]
    sep = "+" + "+".join(["-"*w for _, w in cols]) + "+"

    def row(key, val):
        k = f" {key}".ljust(cols[0][1])
        v = f" {val}".ljust(cols[1][1])
        return f"|{k}|{v}|"

    print(sep)
    print(row("reporte_id", safe_str(rpt.get("_id"))))
    print(sep)
    print(row("usuario_id", safe_str(rpt.get("usuario_id"))))
    print(row("usuario_nombre", safe_str(usuario_nombre)))
    print(row("municipio_id", safe_str(rpt.get("municipio_id"))))
    print(row("municipio_nombre", safe_str(municipio_nombre)))
    print(row("municipio_codigo", safe_str(municipio_codigo)))
    print(row("patente_vehiculo", safe_str(rpt.get("patente_vehiculo"))))
    print(row("estado", safe_str(rpt.get("estado"))))
    print(row("fechaHora_server", safe_str(rpt.get("fechaHora_server"))))
    print(row("ubicacion", safe_str(rpt.get("ubicacion"))))
    print(row("descripcion", safe_str(rpt.get("descripcion"), maxlen=200)))
    print(sep)
    print(row("evidencia_id", safe_str(ev.get("_id") if ev else None)))
    print(row("url_foto", safe_str(ev.get("url_foto") if ev else None)))
    print(row("hash_seguridad_sha", safe_str(ev.get("hash_seguridad_sha") if ev else None)))
    print(row("uploaded_by", safe_str(ev.get("uploaded_by") if ev else None)))
    print(row("uploaded_at", safe_str(ev.get("uploaded_at") if ev else None)))
    print(sep)


+--------------------+----------------------------------------------------------------------+
| reporte_id         | 6a28f45fa8d851a602aefa3a                                             |
+--------------------+----------------------------------------------------------------------+
| usuario_id         | 6a28b11e3ca8ed0b349df8af                                             |
| usuario_nombre     | Elena Paz                                                            |
| municipio_id       | 6a28b11e3ca8ed0b349df8a3                                             |
| municipio_nombre   | Famatina                                                             |
| municipio_codigo   | FAM001                                                               |
| patente_vehiculo   | FAM-ELENA-02                                                         |
| estado             | Pendiente                                                            |
| fechaHora_server   | 2026-06-10 05:21:35.377000           

In [7]:
# Celda 3: Insertar reporte en La Rioja Capital (usuario: Pablo Lopez) — "auto estacionado en porton"
from datetime import datetime, timezone
from bson import ObjectId
from dao.reporte_dao import ReportesDAO

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=80):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

reportes_dao = ReportesDAO(mongo)

# IDs conocidos
USUARIO_ID = ObjectId("6a28b11e3ca8ed0b349df8ac")   # Pablo Lopez (La Rioja Capital)
MUNICIPIO_ID = ObjectId("6a28b11e3ca8ed0b349df8a5") # La Rioja Capital

usuario = mongo.db.usuario.find_one({"_id": USUARIO_ID}, {"_id":1, "municipio_id":1, "nombre_apellido":1})
if not usuario:
    raise RuntimeError("No se encontró el usuario especificado (Pablo Lopez).")

usuario_id = usuario["_id"]
usuario_nombre = usuario.get("nombre_apellido", "NOMBRE_NO_REGISTRADO")
now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": MUNICIPIO_ID,
    "patente_vehiculo": "LRJ-PAB-03",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-66.85, -29.41]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_lrj_pab_003",
    "descripcion": "auto estacionado en porton"
}

evidencia_doc = {
    "url_foto": "https://example.com/foto-porton.jpg",
    "hash_seguridad_sha": "sha256:lrj_pab_003",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

reporte_id, evidencia_id, err = reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)

if err:
    print("Error al insertar:", err)
else:
    rpt = reportes_dao.get_reporte_by_id(reporte_id)
    ev = reportes_dao.evidencia_col.find_one({"reporte_id": reporte_id})
    mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1, "codigo_municipio":1})
    municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"
    municipio_codigo = mun.get("codigo_municipio") if mun else ""

    cols = [("Campo", 20), ("Valor", 70)]
    sep = "+" + "+".join(["-"*w for _, w in cols]) + "+"

    def row(key, val):
        k = f" {key}".ljust(cols[0][1])
        v = f" {val}".ljust(cols[1][1])
        return f"|{k}|{v}|"

    print(sep)
    print(row("reporte_id", safe_str(rpt.get("_id"))))
    print(sep)
    print(row("usuario_id", safe_str(rpt.get("usuario_id"))))
    print(row("usuario_nombre", safe_str(usuario_nombre)))
    print(row("municipio_id", safe_str(rpt.get("municipio_id"))))
    print(row("municipio_nombre", safe_str(municipio_nombre)))
    print(row("municipio_codigo", safe_str(municipio_codigo)))
    print(row("patente_vehiculo", safe_str(rpt.get("patente_vehiculo"))))
    print(row("estado", safe_str(rpt.get("estado"))))
    print(row("fechaHora_server", safe_str(rpt.get("fechaHora_server"))))
    print(row("ubicacion", safe_str(rpt.get("ubicacion"))))
    print(row("descripcion", safe_str(rpt.get("descripcion"), maxlen=200)))
    print(sep)
    print(row("evidencia_id", safe_str(ev.get("_id") if ev else None)))
    print(row("url_foto", safe_str(ev.get("url_foto") if ev else None)))
    print(row("hash_seguridad_sha", safe_str(ev.get("hash_seguridad_sha") if ev else None)))
    print(row("uploaded_by", safe_str(ev.get("uploaded_by") if ev else None)))
    print(row("uploaded_at", safe_str(ev.get("uploaded_at") if ev else None)))
    print(sep)


+--------------------+----------------------------------------------------------------------+
| reporte_id         | 6a28f460a8d851a602aefa3c                                             |
+--------------------+----------------------------------------------------------------------+
| usuario_id         | 6a28b11e3ca8ed0b349df8ac                                             |
| usuario_nombre     | Pablo Lopez                                                          |
| municipio_id       | 6a28b11e3ca8ed0b349df8a5                                             |
| municipio_nombre   | La Rioja Capital                                                     |
| municipio_codigo   | LRJ001                                                               |
| patente_vehiculo   | LRJ-PAB-03                                                           |
| estado             | Pendiente                                                            |
| fechaHora_server   | 2026-06-10 05:21:36.298000           

In [8]:
# Celda: insertar reporte seguro (formato nuevo) y mostrar nombre del usuario y del municipio en tabla
# - Usa ReportesDAO para insertar; muestra salida en tabla clásica
# - Ajustá FORZAR_USUARIO_ID / FORZAR_MUNICIPIO_ID si querés otro usuario/municipio
from datetime import datetime, timezone
from bson import ObjectId
from dao.reporte_dao import ReportesDAO

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=120):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# ---------- Ajustes (usar IDs de tu entorno) ----------
# Municipio Famatina (forzado)
FORZAR_MUNICIPIO_ID = ObjectId("6a28b11e3ca8ed0b349df8a3")
# Usuario del municipio Famatina (ejemplo: Jorge Vera)
FORZAR_USUARIO_ID = ObjectId("6a28b11e3ca8ed0b349df8ae")
# Descripción de ejemplo (cambiar por "auto abandonado" o "auto estacionado en porton" si querés)
DESCRIPCION = "auto estacionado en la vereda"
# URL de evidencia de ejemplo
URL_FOTO = "https://gcdn.emol.cl/argentina/files/2016/05/auto-tapando-rampa.jpg"
# ----------------------------------------------------

# Instanciar DAO
reportes_dao = ReportesDAO(mongo)

# Recuperar usuario (forzado) y asegurarse de obtener el nombre
usuario = mongo.db.usuario.find_one(
    {"_id": FORZAR_USUARIO_ID},
    {"_id": 1, "municipio_id": 1, "nombre_apellido": 1}
)
if not usuario:
    raise RuntimeError("No se encontró el usuario forzado. Verificá FORZAR_USUARIO_ID.")

usuario_id = usuario["_id"]
usuario_nombre = usuario.get("nombre_apellido", "NOMBRE_NO_REGISTRADO")

# Usar municipio forzado (Famatina)
municipio_id = FORZAR_MUNICIPIO_ID

now = now_utc()

# Documento de reporte (adaptado al formato nuevo)
reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "FAM-DEMO-01",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-67.85, -29.21]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_fam_demo_001",
    "descripcion": DESCRIPCION
}

# Documento de evidencia
evidencia_doc = {
    "url_foto": URL_FOTO,
    "hash_seguridad_sha": "sha256:fam_demo_001",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Insertar usando el DAO (maneja validaciones/rollback según implementación)
reporte_id, evidencia_id, err = reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)

if err:
    print("Error al insertar:", err)
else:
    # Recuperar documentos para mostrar
    rpt = reportes_dao.get_reporte_by_id(reporte_id)
    ev = reportes_dao.evidencia_col.find_one({"reporte_id": reporte_id})
    mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1, "provincia":1, "pais":1, "codigo_municipio":1, "contacto":1})
    municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"
    municipio_codigo = mun.get("codigo_municipio") if mun else mun.get("codigo") if mun else ""

    # Preparar tabla clásica (anchos ajustables)
    cols = [("Campo", 20), ("Valor", 70)]
    sep = "+" + "+".join(["-"*w for _, w in cols]) + "+"

    def row(key, val):
        k = f" {key}".ljust(cols[0][1])
        v = f" {val}".ljust(cols[1][1])
        return f"|{k}|{v}|"

    print(sep)
    print(row("reporte_id", safe_str(rpt.get("_id"))))
    print(sep)
    print(row("usuario_id", safe_str(rpt.get("usuario_id"))))
    print(row("usuario_nombre", safe_str(usuario_nombre)))
    print(row("municipio_id", safe_str(rpt.get("municipio_id"))))
    print(row("municipio_nombre", safe_str(municipio_nombre)))
    print(row("municipio_codigo", safe_str(municipio_codigo)))
    print(row("patente_vehiculo", safe_str(rpt.get("patente_vehiculo"))))
    print(row("estado", safe_str(rpt.get("estado"))))
    print(row("fechaHora_server", safe_str(rpt.get("fechaHora_server"))))
    print(row("ubicacion", safe_str(rpt.get("ubicacion"))))
    print(row("descripcion", safe_str(rpt.get("descripcion"), maxlen=200)))
    print(sep)
    print(row("evidencia_id", safe_str(ev.get("_id") if ev else None)))
    print(row("url_foto", safe_str(ev.get("url_foto") if ev else None)))
    print(row("hash_seguridad_sha", safe_str(ev.get("hash_seguridad_sha") if ev else None)))
    print(row("uploaded_by", safe_str(ev.get("uploaded_by") if ev else None)))
    print(row("uploaded_at", safe_str(ev.get("uploaded_at") if ev else None)))
    print(sep)


+--------------------+----------------------------------------------------------------------+
| reporte_id         | 6a28f461a8d851a602aefa3e                                             |
+--------------------+----------------------------------------------------------------------+
| usuario_id         | 6a28b11e3ca8ed0b349df8ae                                             |
| usuario_nombre     | Jorge Vera                                                           |
| municipio_id       | 6a28b11e3ca8ed0b349df8a3                                             |
| municipio_nombre   | Famatina                                                             |
| municipio_codigo   | FAM001                                                               |
| patente_vehiculo   | FAM-DEMO-01                                                          |
| estado             | Pendiente                                                            |
| fechaHora_server   | 2026-06-10 05:21:37.587000           

### 📊 Listar reportes agrupados por municipio

Muestra la cantidad de reportes y un resumen compacto (ID corto, patente, estado, fecha, descripción) usando `ReportesDAO.list_recent_grouped_by_municipio`.

* **Propósito:** Útil para la presentación (verificar rápidamente municipio y cantidad).
* ⚠️ **Nota:** Asegúrate de haber ejecutado antes las celdas de inicialización (`MongoDAO` y `ReportesDAO`).

In [9]:
# Celda: mostrar reportes agrupados por municipio (usa ReportesDAO.list_recent_grouped_by_municipio)
# Ejecutar solo esta celda para ver la tabla de reportes recientes
import importlib
import dao.reporte_dao as rd
importlib.reload(rd)
from dao.reporte_dao import ReportesDAO
from datetime import datetime

MAX_REPORTS = 200

def short_id(oid, n=6):
    s = str(oid) if oid is not None else ""
    return s[:n]

def fmt_date(d):
    return d.isoformat(sep=" ") if d is not None else ""

def safe_trunc(s, maxlen=60):
    s = str(s or "")
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# Instanciar DAO
reportes_dao = ReportesDAO(mongo)

# Obtener datos agrupados desde el DAO
grouped = reportes_dao.list_recent_grouped_by_municipio(limit=MAX_REPORTS)

if not grouped:
    print("No hay reportes para mostrar.")
else:
    for mun, rep_list in grouped:
        print("="*100)
        print(f"Municipio: {mun}  —  Reportes: {len(rep_list)}")
        print("-"*100)

        headers = ["ID", "Patente", "Estado", "Fecha", "Descripción"]
        col_widths = {h: len(h) for h in headers}
        for r in rep_list:
            col_widths["ID"] = max(col_widths["ID"], len(short_id(r.get("_id"), 6)))
            col_widths["Patente"] = max(col_widths["Patente"], len(str(r.get("patente_vehiculo") or "")))
            col_widths["Estado"] = max(col_widths["Estado"], len(str(r.get("estado") or "")))
            col_widths["Fecha"] = max(col_widths["Fecha"], len(fmt_date(r.get("fechaHora_server"))))
            col_widths["Descripción"] = max(col_widths["Descripción"], min(80, len((r.get("descripcion") or "")[:160])))

        # Limitar anchos razonables
        for k in col_widths:
            if col_widths[k] > 80:
                col_widths[k] = 80

        sep = " | "
        header_line = sep.join(h.ljust(col_widths[h]) for h in headers)
        divider = "-+-".join("-" * col_widths[h] for h in headers)
        print(header_line)
        print(divider)

        for r in rep_list:
            fid = short_id(r.get("_id"), 6).ljust(col_widths["ID"])
            patente = (r.get("patente_vehiculo") or "").ljust(col_widths["Patente"])
            estado = (r.get("estado") or "").ljust(col_widths["Estado"])
            fecha = fmt_date(r.get("fechaHora_server")).ljust(col_widths["Fecha"])
            desc = safe_trunc((r.get("descripcion") or ""), col_widths["Descripción"]).ljust(col_widths["Descripción"])
            print(f"{fid} | {patente} | {estado} | {fecha} | {desc}")
        print()


Municipio: Chilecito  —  Reportes: 18
----------------------------------------------------------------------------------------------------
ID     | Patente | Estado    | Fecha                      | Descripción                                                               
-------+---------+-----------+----------------------------+---------------------------------------------------------------------------
6a28f4 | DEMO123 | Pendiente | 2026-06-10 05:21:32.908000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta como evidencia.
6a28eb | DEMO123 | Pendiente | 2026-06-10 04:45:41.502000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta como evidencia.
6a28d1 | DEMO123 | Pendiente | 2026-06-10 02:51:09.624000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta como evidencia.
6a28be | DEMO123 | Pendiente | 2026-06-10 01:32:09.350000 | Prueba desde notebook usando ReportesDAO.                                 
6a28be | DEMO123 | Pendiente | 2026-06-10 01:31:44.

### 🚫 Prueba: Intentar insertar un reporte sin evidencia

**Qué hace:** Intenta registrar un reporte omitiendo la evidencia para demostrar que las reglas de negocio del `DAO` bloquean la operación correctamente.

* **Objetivo:** Verificar el manejo de excepciones y la validación estricta de la capa de persistencia.

In [10]:
# Celda: intentar insertar un REPORTE SIN EVIDENCIA y mostrar que la regla del DAO lo bloquea
# Ejecutar solo esta celda para comprobar que la regla "no insertar reportes sin evidencia" está activa.
import importlib
import dao.reporte_dao as rd
importlib.reload(rd)
from dao.reporte_dao import ReportesDAO
from bson import ObjectId
from datetime import datetime, timezone

def now_utc():
    return datetime.now(timezone.utc)

def banner(msg, kind="ERROR"):
    line = "=" * (len(msg) + 8)
    tag = "!!!" if kind == "ERROR" else "***"
    print(f"\n{line}\n{tag}  {msg}  {tag}\n{line}\n")

# Ajustá estos IDs si necesitás (ejemplo: usuario de Famatina)
USUARIO_ID = ObjectId("6a28b11e3ca8ed0b349df8ae")   # Jorge Vera (Famatina)
MUNICIPIO_ID = ObjectId("6a28b11e3ca8ed0b349df8a3") # Famatina

# Reporte de prueba SIN evidencia
now = now_utc()
reporte_sin_evidencia = {
    "usuario_id": USUARIO_ID,
    "municipio_id": MUNICIPIO_ID,
    "patente_vehiculo": "NOEV-TEST-001",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type":"Point","coordinates":[-67.85,-29.21]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_no_evid_test",
    "descripcion": "Prueba: intento crear reporte SIN evidencia (debe bloquearse por regla de negocio)."
}

# Instanciar DAO y probar create_reporte (que ahora exige evidencia)
dao = ReportesDAO(mongo)

banner("INTENTO: crear reporte SIN evidencia", kind="ERROR")
try:
    # create_reporte ahora exige evidencia y debe lanzar ValueError (o excepción controlada)
    dao.create_reporte(dict(reporte_sin_evidencia))
    # Si llegamos aquí, la regla no se aplicó correctamente
    banner("INSERCIÓN INESPERADA", kind="ERROR")
    print("Se insertó un reporte sin evidencia. Esto indica que la regla de negocio NO está activa en el DAO.")
    print("Revisá dao/reporte_dao.py para asegurarte de que create_reporte exige evidencia o usá insert_report_with_evidence.")
except Exception as e:
    # Mensaje claro y orientador
    banner("BLOQUEADO: no se permite crear reportes sin evidencia", kind="ERROR")
    print("Motivo (excepción lanzada por el DAO):", type(e).__name__, str(e))
    print("\nComportamiento esperado: el DAO impide la inserción y devuelve un error claro.")
    print("Acción recomendada: crear el reporte usando el método seguro del DAO con evidencia:")
    print("  reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)")



!!!  INTENTO: crear reporte SIN evidencia  !!!


!!!  BLOQUEADO: no se permite crear reportes sin evidencia  !!!

Motivo (excepción lanzada por el DAO): ValueError No se permite insertar un reporte sin evidencia. Usá insert_report_with_evidence(reporte_doc, evidencia_doc).

Comportamiento esperado: el DAO impide la inserción y devuelve un error claro.
Acción recomendada: crear el reporte usando el método seguro del DAO con evidencia:
  reportes_dao.insert_report_with_evidence(reporte_doc, evidencia_doc)
